# Arabic OCR Post-Correction

Finetunes Qwen2.5-0.5B to repair Arabic OCR output.

[Code](https://github.com/Crypto47/arabic-ocr-post-correction) · [Model](https://huggingface.co/Sheeda/arabic-ocr-post-correction-0.5b) · Requires a GPU runtime.

Check the GPU.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

Install dependencies.

In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.14" "trl>=0.20" \
                "datasets>=2.20" "accelerate>=0.33" pyyaml kagglehub

Clone the repo and enter it.

In [ ]:
import os
import pathlib
import subprocess

REPO_URL = "https://github.com/Crypto47/arabic-ocr-post-correction.git"
NAME = "arabic-ocr-post-correction"

hits = [q for q in pathlib.Path("/content").rglob(NAME) if (q / "src").is_dir()]
if hits:
    PROJECT = str(hits[0])
    subprocess.run(["git", "-C", PROJECT, "pull", "-q"], check=False)
else:
    PROJECT = "/content/" + NAME
    subprocess.run(["git", "clone", "-q", REPO_URL, PROJECT], check=True)

os.chdir(PROJECT)
print("PROJECT =", PROJECT)

Authenticate with Kaggle.

In [ ]:
import kagglehub

try:
    print("signed in as:", kagglehub.whoami()["username"])
except Exception:
    kagglehub.login()

Download and unpack the Arabic corpus.

In [ ]:
DATASET = "mohamedbentalb/arasum"

DATA_DIR = kagglehub.dataset_download(DATASET)
print(DATA_DIR)

Report what the download contains.

In [ ]:
import collections
from pathlib import Path

TEXT = {".txt", ".jsonl", ".json", ".csv", ".tsv", ".parquet"}

root = Path(DATA_DIR)
files = [p for p in root.rglob("*") if p.is_file()]
counts = collections.Counter(p.suffix.lower() or "(none)" for p in files)

print(f"{len(files)} files, {sum(p.stat().st_size for p in files) / 1e6:.0f} MB")
for ext, n in counts.most_common(10):
    print(f"  {ext:<12} {n}")
print("text files present:", bool(set(counts) & TEXT))

Generate (noisy → clean) training pairs.

In [ ]:
!cd "{PROJECT}" && python src/build_dataset.py \
    --input "{DATA_DIR}" \
    --output data/processed \
    --max-pairs 60000 \
    --min-rate 0.04 --max-rate 0.18

Preview three pairs.

In [ ]:
import json

with open(f"{PROJECT}/data/processed/train.jsonl", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        r = json.loads(line)
        print("NOISY:", r["messages"][0]["content"].split("\n\n")[-1])
        print("CLEAN:", r["messages"][1]["content"])
        print()

Score the untuned base model.

In [ ]:
!cd "{PROJECT}" && python src/evaluate.py --limit 200 --out results/eval_base.json

Train.

In [ ]:
!cd "{PROJECT}" && python src/train.py --config configs/qwen05b_lora.yaml

Score the finetuned model.

In [ ]:
!cd "{PROJECT}" && python src/evaluate.py \
    --adapter outputs/arabic-ocr-post-correction/final \
    --limit 200 \
    --out results/eval_tuned.json \
    --save-predictions results/preds.jsonl

Score it again with the drift guardrail.

In [ ]:
!cd "{PROJECT}" && python src/evaluate.py \
    --adapter outputs/arabic-ocr-post-correction/final \
    --limit 200 --max-drift 0.20 \
    --out results/eval_guarded.json

Compare all four.

In [ ]:
import json

base = json.load(open(f"{PROJECT}/results/eval_base.json", encoding="utf-8"))
tuned = json.load(open(f"{PROJECT}/results/eval_tuned.json", encoding="utf-8"))
guard = json.load(open(f"{PROJECT}/results/eval_guarded.json", encoding="utf-8"))

print(f"{'':<22}{'CER':>10}{'WER':>10}")
print("-" * 42)
print(f"{'raw OCR':<22}{base['baseline']['cer']:>10.4f}{base['baseline']['wer']:>10.4f}")
print(f"{'base 0.5B':<22}{base['corrected']['cer']:>10.4f}{base['corrected']['wer']:>10.4f}")
print(f"{'finetuned':<22}{tuned['corrected']['cer']:>10.4f}{tuned['corrected']['wer']:>10.4f}")
print(f"{'finetuned + guard':<22}{guard['corrected']['cer']:>10.4f}{guard['corrected']['wer']:>10.4f}")

Correct one sample.

In [ ]:
!cd "{PROJECT}" && python src/infer.py \
    --adapter outputs/arabic-ocr-post-correction/final \
    --text "العلم نور يضنء طزيق الإنسان في الحتياة، وآلجهل ظلام دام"

Sign in to HuggingFace.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

Publish the adapter.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

REPO_ID = "Sheeda/arabic-ocr-post-correction-0.5b"
BASE = "Qwen/Qwen2.5-0.5B-Instruct"
ADAPTER = f"{PROJECT}/outputs/arabic-ocr-post-correction/final"

model = PeftModel.from_pretrained(AutoModelForCausalLM.from_pretrained(BASE), ADAPTER)
model.push_to_hub(REPO_ID)
AutoTokenizer.from_pretrained(ADAPTER).push_to_hub(REPO_ID)
print(f"https://huggingface.co/{REPO_ID}")